# Tab-pretext split experiment — does the weak group improve with a dedicated model?
The joint 16-feature model (`tab-mdn-v1`) predicts magnitudes/sizes well but is weak on
**u−g (R² 0.59), conc_r (0.71), i−z (0.73), petroR90 (0.83), petroRad (0.84), r−i (0.86)**.
Hypothesis: the easy features dominate the shared embedding; a model trained **only on the
weak group** may predict them better.

Trains TWO models — `tab-mdn-hard-v1` (6 weak features) and `tab-mdn-easy-v1` (10 strong
features) — then compares per-feature R² on the v4-5 val against the joint model's numbers.

Runtime → Change runtime type → **GPU**.


In [ ]:
import os
os.environ['CUTOUT_SIZE'] = '24'        # sample_v4.5 is 24x24
![ -d /content/Photo-Z-CNN-Model ] || git clone -b main https://github.com/zhhrozhh/Photo-Z-CNN-Model.git /content/Photo-Z-CNN-Model
%cd /content/Photo-Z-CNN-Model
!pip -q install mlflow


In [ ]:
# Google auth — if 'credential propagation was unsuccessful', just re-run this cell
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q


In [ ]:
!mkdir -p /content/data
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.5/catalog_v4.parquet /content/data/catalog_v1.parquet
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.5/images_*.npy /content/data/
DATA_DIR = '/content/data'

import tensorflow as tf
print('GPUs =', tf.config.list_physical_devices('GPU'))


In [ ]:
try:
    from google.colab import userdata
    MLFLOW_TOKEN = userdata.get('MLFLOW_TOKEN')
except Exception as e:
    print('no MLFLOW_TOKEN secret ->', e); MLFLOW_TOKEN = None


## Feature groups + joint-model reference (R² on v4-5 val, run `tab-mdn-v1`)


In [ ]:
HARD = ['u-g', 'r-i', 'i-z', 'log_petroRad_r', 'log_petroR90_r', 'conc_r']   # R2 < 0.87 jointly
EASY = ['dered_u', 'dered_g', 'dered_r', 'dered_i', 'dered_z', 'g-r',
        'log_expRad_r', 'log_deVRad_r', 'log_petroR50_r', 'fracDeV_r']

JOINT_R2 = {  # reference: the 16-feature joint model, same val set
    'dered_u': .937, 'dered_g': .962, 'dered_r': .933, 'dered_i': .970, 'dered_z': .925,
    'g-r': .962, 'u-g': .588, 'r-i': .861, 'i-z': .728,
    'log_expRad_r': .948, 'log_deVRad_r': .943, 'log_petroRad_r': .835,
    'log_petroR50_r': .891, 'log_petroR90_r': .828, 'fracDeV_r': .879, 'conc_r': .713}


## Train the two models


In [ ]:
from tab_cnn import train_tab

hist_h, model_hard, (fm_h, fs_h) = train_tab(
    data_dir=DATA_DIR, crop=24, preproc='p99', features=HARD,
    N=None, batch=256, lr=3e-4, epochs=50,
    run_name='tab-mdn-hard-v1', mlflow_token=MLFLOW_TOKEN)


In [ ]:
hist_e, model_easy, (fm_e, fs_e) = train_tab(
    data_dir=DATA_DIR, crop=24, preproc='p99', features=EASY,
    N=None, batch=256, lr=3e-4, epochs=50,
    run_name='tab-mdn-easy-v1', mlflow_token=MLFLOW_TOKEN)


## Compare per-feature R² — dedicated vs joint


In [ ]:
import glob, re
import numpy as np, pandas as pd
import eval as ev, fusion as fu
from tab_cnn import tab_targets

cat, z_all, o2i, Y = tab_targets(DATA_DIR)
va = np.array([o2i[int(o)] for o in pd.read_csv('splits/v4-5-validate.csv').objid if int(o) in o2i])

SHARD = 6000
mm = {int(re.findall(r'images_(\d+)_', p)[0]) // SHARD: np.load(p, mmap_mode='r')
      for p in glob.glob(f'{DATA_DIR}/images_*.npy')}
X = np.empty((len(va), 24, 24, 5), np.float16)
for s in np.unique(va // SHARD):
    sel = np.where(va // SHARD == s)[0]; rr = va[sel] % SHARD; srt = np.argsort(rr)
    X[sel[srt]] = mm[int(s)][rr[srt]]
pp = ev.make_np_preprocess('p99')
Xp = pp(np.asarray(X, 'float32'))

def per_feature_r2(model, feats, fm, fs):
    raw = model.predict(Xp, batch_size=1024, verbose=0)          # (n, F, 6)
    pi, mu = raw[..., :2], raw[..., 2:4]
    pred = (pi * mu).sum(-1) * fs + fm
    out = {}
    for j, name in enumerate(feats):
        i = fu.TAB_FEATURES.index(name)
        m = np.isfinite(Y[va, i])
        resid = pred[m, j] - Y[va, i][m]
        out[name] = 1 - np.var(resid) / np.var(Y[va, i][m])
    return out

r2_h = per_feature_r2(model_hard, HARD, fm_h, fs_h)
r2_e = per_feature_r2(model_easy, EASY, fm_e, fs_e)

print(f"{'feature':<16} {'joint':>7} {'dedicated':>10} {'delta':>7}")
for grp, r2s in [('HARD', r2_h), ('EASY', r2_e)]:
    print(f'--- {grp} ---')
    for name, v in r2s.items():
        print(f'{name:<16} {JOINT_R2[name]:7.3f} {v:10.3f} {v - JOINT_R2[name]:+7.3f}')


## Save embeddings (both models) for HGB stacking at home


In [ ]:
import tensorflow as tf
for tag, model in [('hard', model_hard), ('easy', model_easy)]:
    embn = tf.keras.Model(model.input, model.get_layer('embedding').output)
    paths = sorted(glob.glob(f'{DATA_DIR}/images_*.npy'), key=lambda p: int(re.findall(r'images_(\d+)_', p)[0]))
    emb = np.concatenate([embn.predict(pp(np.asarray(np.load(p), 'float32')), batch_size=1024, verbose=0)
                          for p in paths])
    np.save(f'/content/emb_tabmdn_{tag}_v1.npy', emb)
    print(tag, emb.shape)
!gcloud storage cp /content/emb_tabmdn_*_v1.npy gs://macrocosm-lewagon/results/
